## 基于MindSpore框架的UNet-2D案例实现

### 1 模型简介

Unet模型于2015年在论文《U-Net: Convolutional Networks for Biomedical Image Segmentation》中被提出，最初的提出是为了解决医学图像分割问题，用于细胞层面的图像分割任务。

Unet模型是在FCN网络的基础上构建的，但由于FCN无法获取上下文信息以及位置信息，导致准确性较低，Unet模型由此引入了U型结构获取上述两种信息，并且模型结构简单高效、容易构建，在较小的数据集上也能实现较高的准确率。

#### 1.1 模型结构
Unet模型的整体结构由两部分组成，即特征提取网络和特征融合网络，其结构也被称为“编码器-解码器结构”，并且由于网络整体结构类似于大写的英文字母“U”，故得名Unet，在其原始论文中定义的网络结构如图1所示。

<center>
    <img src="https://tva1.sinaimg.cn/large/e6c9d24ely1h5bw5io8tqj20n10fa3zo.jpg" alt="image-20220819101847606" style="zoom:67%;" />
    <br>
    <div style="color:orange;
    display: inline-block;
    color: #999;
    padding: 2px;">图1 Unet网络结构图</div>
</center>

整个模型结构就是在原始图像输入后，首先进行特征提取，再进行特征融合：

a) 左半部分负责特征提取的网络结构（即编码器结构）需要利用两个3x3的卷积核与2x2的池化层组成一个“下采样模块”，每一个下采样模块首先会对特征图进行两次valid卷积，再进行一次池化操作。由此经过4个下采样模块后，原始尺寸为572x572大小、通道数为1的原始图像，转换为了大小为28x28、通道数为1024的特征图。

b) 右半部分负责进行上采样的网络结构（即解码器结构）需要利用1次反卷积操作、特征拼接操作以及两个3x3的卷积核作为一个“上采样模块”，每一个上采样模块首先会对特征图通过反卷积操作使图像尺寸增加1倍，再通过拼接编码器结构中的特征图使得通道数增加，最后经过两次valid卷积。由此经过4个上采样模块后，经过下采样模块的、大小为28x28、通道数为1024的特征图，转换为了大小为388x388、通道数为64的特征图。

c) 网络结构的最后一部分是通过两个1x1的卷积核将经过上采样得到的通道数为64的特征图，转换为了通道数为2的图像作为预测结果输出。
#### 1.2 模型特点

a) 利用拼接操作将低级特征图与高级特征图进行特征融合

b) 完全对称的U型结构使得高分辨率信息和低分辨率信息在目标图片中增加，前后特征融合更为彻底。

c) 结合了下采样时的低分辨率信息（提供物体类别识别依据）和上采样时的高分辨率信息（提供精准分割定位依据），此外还通过融合操作填补底层信息以提高分割精度。


### 2 案例实现

#### 2.1 **环境准备与数据读取**

本案例基于MindSpore-Ascend版本实现，在Ascend上完成模型训练。

案例实现所使用的数据即ISBI果蝇电镜图数据集，直接从本地加载数据，在data目录下载好的数据集包括3个tif文件，分别对应测试集样本、训练集标签、训练集样本，文件路径结构如下：

```
.datasets/
└── ISBI
    ├── test-volume.tif
    ├── train-labels.tif
    └── train-volume.tif
```

其中每个tif文件都由30副图片压缩而成，所以接下来需要获取每个tif文件中所存储的所有图片，将其转换为png格式存储，得到训练集样本对应的30张png图片、训练集标签对应的30张png图片以及测试集样本对应的30张png图片。

In [ ]:
# 需要Ascend版本的mindspore，因为cpu版本目前不支持mint.nn.conv2d
! pip install ml_collections
! pip install numpy
! pip install matplotlib

In [ ]:
import sys
import mindspore
print(sys.executable)
mindspore.set_device('Ascend')
print(mindspore.run_check())
from PIL import Image, ImageSequence
import math
import numpy as np
import matplotlib.pyplot as plt
#显示下载好的数据
train_image_path = "data/train-volume.tif"
train_masks_path = "data/train-labels.tif"
image = np.array([np.array(p) for p in ImageSequence.Iterator(Image.open(train_image_path))])
masks = np.array([np.array(p) for p in ImageSequence.Iterator(Image.open(train_masks_path))])

def show_image(image_list,num = 6):
    img_titles = []
    img_draws = []
    for ind,img in enumerate(image_list):
        if ind == num:
            break
        img_titles.append(ind)
        img_draws.append(img)

    for i in range(len(img_titles)):
        if len(img_titles) > 6:
            row = 3
        elif 3<len(img_titles)<=6:
            row = 2
        else:
            row = 1
        col = math.ceil(len(img_titles)/row)
        plt.subplot(row,col,i+1),plt.imshow(img_draws[i],'gray')
        plt.title(img_titles[i])
        plt.xticks([]),plt.yticks([])
    plt.show()
    
show_image(image,num = 12)
show_image(masks,num = 12)

具体的实现方式首先是将tif文件转换为数组形式，之后通过io操作将每张图片对应的数组存储为png图像，处理过后的训练集样本及其对应的标签图像如图2所示。将3个tif文件转换为png格式后，针对训练集的样本与标签，将其以2:1的比例，重新划分为了训练集与验证集，划分完成后的文件路径结构如下：

```
.datasets/
└── ISBI
    ├── test_imgs
    │   ├── 00000.png
    │   ├── 00001.png
    │   └── . . . . .
    ├── train
    │   ├── image
    │   │   ├── 00001.png
    │   │   ├── 00002.png
    │   │   └── . . . . .
    │   └── mask
    │       ├── 00001.png
    │       ├── 00002.png
    │       └── . . . . .
    └── val
        ├── image
        │   ├── 00000.png
        │   ├── 00003.png
        │   └── . . . . .
        └── mask
            ├── 00000.png
            ├── 00003.png
            └── . . . . .
```

<center>
    <img src="https://tva1.sinaimg.cn/large/e6c9d24ely1h5bwhda4g2j20fz0chabb.jpg" alt="image-20220819101847606" style="zoom:50%;" />
    <br>
    <div style="color:orange;
    display: inline-block;
    color: #999;
    padding: 2px;">图2 训练集样本及其对应标签</div>
</center>

#### 2.2 数据集创建

在进行上述tif文件格式转换，以及测试集和验证集的进一步划分后，就完成了数据读取所需的所有工作，接下来就需要利用处理好的图像数据，通过一定的图像变换来进行数据增强，并完成数据集的创建。

数据增强部分是引入了mindspore.dataset.vision，针对训练集样本和标签，首先通过A.resize()方法将图像尺寸重新调整为统一大小，之后再进行转置以及水平翻转、垂直翻转，完成针对训练集样本和标签的数据增强。针对验证集的样本和标签，仅通过resize()方法将图像尺寸重新调整为统一大小。

其次数据集的创建部分，首先是定义了Data_Loader类，在该类的__init__函数中，根据传入的data_path参数，确定在数据读取阶段设置好的、训练集和验证集的存储路径，再设置对应的样本和标签路径，并针对训练集和验证集的不同数据增强方法。在该类的__getitem__函数中，通过传入索引值读取训练集或验证集存储路径下的样本和标签图像，并对图像进行对应的数据增强操作，之后再对样本和标签的形状进行转置，就完成了__getitem__函数对样本和标签图像的读取。最后通过定义create_dataset函数，传入data_dir、batch_size等参数，在函数中实例化Data_Loader类获取data_dir，也就是训练集或验证集对应路径下的样本和标签元组对，再通过mindspore.dataset中的GeneratorDataset将元组转换为Tensor，最后通过设定好的batch_size将样本和标签按照batch_size大小分组，由此完成数据集的创建，上述流程对应代码如下：

In [ ]:
# 划分数据集 tif->image
import os
import cv2
import numpy as np

# 1. 定义保存路径 
root_path = 'datasets/ISBI'
train_img_dir = os.path.join(root_path, 'train', 'image')
train_mask_dir = os.path.join(root_path, 'train', 'mask')
val_img_dir = os.path.join(root_path, 'val', 'image')
val_mask_dir = os.path.join(root_path, 'val', 'mask')

# 2. 创建文件夹
for d in [train_img_dir, train_mask_dir, val_img_dir, val_mask_dir]:
    os.makedirs(d, exist_ok=True)

print(f"正在转换数据... 总帧数: {len(image)}")

# 3. 开始切分并保存
# 2:1的方式
split_point = 20

for i in range(len(image)):
    # 获取单帧数据
    img_frame = image[i]
    mask_frame = masks[i]
    
    # 决定存放在 训练集 还是 验证集
    if i < split_point:
        save_img_path = os.path.join(train_img_dir, f"{i}.png")
        save_mask_path = os.path.join(train_mask_dir, f"{i}.png")
    else:
        save_img_path = os.path.join(val_img_dir, f"{i}.png")
        save_mask_path = os.path.join(val_mask_dir, f"{i}.png")
    
    # 保存图片 
    cv2.imwrite(save_img_path, img_frame)
    cv2.imwrite(save_mask_path, mask_frame)

print("转换完成！")
print(f"训练集保存路径: {train_img_dir}")
print(f"验证集保存路径: {val_img_dir}")

In [ ]:
import os
import cv2
import mindspore.dataset as ds
import glob
import mindspore.dataset.vision as vision_C  #.c_transforms
import mindspore.dataset.transforms as C_transforms #.c_transform
import random
import mindspore
from mindspore.dataset.vision import Inter

def train_transforms(img_size):
    return [
    vision_C.Resize(img_size, interpolation=Inter.NEAREST),
    vision_C.Rescale(1./255., 0.0),
    vision_C.RandomHorizontalFlip(prob=0.5),
    vision_C.RandomVerticalFlip(prob=0.5),
    vision_C.HWC2CHW()
    ]


def val_transforms(img_size):
    return [
    vision_C.Resize(img_size, interpolation=Inter.NEAREST),
    vision_C.Rescale(1/255., 0),
    vision_C.HWC2CHW()
    ]



class Data_Loader:
    def __init__(self, data_path):
        # 初始化函数，读取所有data_path下的图片
        self.data_path = data_path
        self.imgs_path = glob.glob(os.path.join(data_path, 'image/*.png'))
        self.label_path = glob.glob(os.path.join(data_path, 'mask/*.png'))

    def __getitem__(self, index):
        # 根据index读取图片
        image = cv2.imread(self.imgs_path[index])
        label = cv2.imread(self.label_path[index], cv2.IMREAD_GRAYSCALE)
        label = label.reshape((label.shape[0], label.shape[1], 1))
    
        return image, label

    @property
    def column_names(self):
        column_names = ['image', 'label']
        return column_names

    def __len__(self):
        # 返回训练集大小
        return len(self.imgs_path)


def create_dataset(data_dir, img_size, batch_size, augment, shuffle):
    mc_dataset = Data_Loader(data_path=data_dir)
    dataset = ds.GeneratorDataset(mc_dataset, mc_dataset.column_names, shuffle=shuffle)

    if augment:
        transform_img = train_transforms(img_size)
    else:
        transform_img = val_transforms(img_size)

    seed = random.randint(1,1000)
    mindspore.set_seed(seed)
    dataset = dataset.map(input_columns='image', num_parallel_workers=1, operations=transform_img)
    mindspore.set_seed(seed)
    dataset = dataset.map(input_columns="label", num_parallel_workers=1, operations=transform_img)

    if shuffle:
        dataset = dataset.shuffle(buffer_size=10000)
    dataset = dataset.batch(batch_size, num_parallel_workers=1)
    if augment == True and shuffle == True:
        print("训练集数据量：", len(mc_dataset))
    elif augment == False and shuffle == False:
        print("验证集数据量：", len(mc_dataset))
    else:
        pass
    return dataset

In [ ]:
if __name__ == '__main__':
    train_dataset = create_dataset('datasets/ISBI/val', img_size=224, batch_size=3, augment=False, shuffle=False)
    for item, (image, label) in enumerate(train_dataset):
        if item < 5:
            print(f"Shape of image [N, C, H, W]: {image.shape} {image.dtype}",'---',f"Shape of label [N, C, H, W]: {label.shape} {label.dtype}")

#### 2.3 模型构建

本案例实现中所构建的Unet模型结构与2015年论文中提出的Unet结构大致相同，但本案例中Unet网络模型的“下采样模块”与“上采样模块”使用的卷积类型都为Same卷积，而原论文中使用的是Valid卷积。此外，原论文的网络模型最终使用两个1x1的卷积核，输出了通道数2的预测图像，而本案例的网络模型最终使用的是1个1x1的卷积核，输出通道数为1的灰度图，和标签图像格式保持一致。实际构建的Unet模型结构如图3所示。

<center>
    <img src="https://tva1.sinaimg.cn/large/e6c9d24ely1h5bwhpesqpj20no0g7jso.jpg" alt="image-20220819101847606" style="zoom:67%;" />
    <br>
    <div style="color:orange;
    display: inline-block;
    color: #999;
    padding: 2px;">图3 实际构建的Unet模型结构</div>
</center>

MindSpore框架构建网络的流程与PyTorch类似，在定义模型类时需要继承Cell类，并重写__init__和construct方法。具体的实现方式首先是定义了一个DoubleConv模型类，在类中重写__init__方法，通过使用mint.nn.Conv2d层定义“下采样模块”与“上采样模块”中都使用到的两个卷积函数，并且在每个卷积层后加入mint.nn.BatchNorm2d层来对每次卷积后的特征图进行标准化，防止过拟合，以及使用mint.nn.ReLU层加入非线性的激活函数。之后在construct方法中使用定义好的运算构建前向网络。

在DoubleConv模型类定义好之后，接下来就是通过定义UNet模型类来完成整个UNet网络的构建。在UNet模型类的__init__方法中实例化double_conv类来表示两个连续的卷积层，接着使用mint.nn.MaxPool2d来进行最大池化，由此完成了1个“下采样模块”的构建，重复4次即可完成网络中的编码器部分。针对解码器部分，使用了mint.nn.ResizeBilinear层来表示反卷积层，接着实例化了DoubleConv类来表示两个卷积层，由此完成了1个“上采样模块”的构建，重复4次即完成网络中解码器部分的搭建。之后通过1个mint.nn.Conv2d层来完成预测图像的输出。最后在construct方法中使用定义好的运算构建前向网络，由此完成整个Unet网络模型的构建。上述构建流程的对应代码如下所示：


In [ ]:
import mindspore as ms
from mindspore import nn, mint

# DoubleConv 
class DoubleConv(nn.Cell):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.seq = nn.SequentialCell(
            mint.nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            mint.nn.BatchNorm2d(out_ch),
            mint.nn.ReLU(),
            mint.nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            mint.nn.BatchNorm2d(out_ch),
            mint.nn.ReLU()
        )

    def construct(self, x):
        return self.seq(x)

# 修改 UNet 类
class UNet(nn.Cell):
    def __init__(self, in_ch=3, n_classes=1):
        super(UNet, self).__init__()
        
        # --- 下采样部分 ---
        self.double_conv1 = DoubleConv(in_ch, 64)
        self.maxpool1 = mint.nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.double_conv2 = DoubleConv(64, 128)
        self.maxpool2 = mint.nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.double_conv3 = DoubleConv(128, 256)
        self.maxpool3 = mint.nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.double_conv4 = DoubleConv(256, 512)
        self.maxpool4 = mint.nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.double_conv5 = DoubleConv(512, 1024)

        # --- 上采样部分 ---
        
        self.double_conv6 = DoubleConv(1024 + 512, 512)
        self.double_conv7 = DoubleConv(512 + 256, 256)
        self.double_conv8 = DoubleConv(256 + 128, 128)
        self.double_conv9 = DoubleConv(128 + 64, 64)

        self.final = mint.nn.Conv2d(64, n_classes, kernel_size=1)

    def construct(self, x):
        # 编码器 (Encoder)
        feature1 = self.double_conv1(x)
        tmp = self.maxpool1(feature1)
        
        feature2 = self.double_conv2(tmp)
        tmp = self.maxpool2(feature2)
        
        feature3 = self.double_conv3(tmp)
        tmp = self.maxpool3(feature3)
        
        feature4 = self.double_conv4(tmp)
        tmp = self.maxpool4(feature4)
        
        feature5 = self.double_conv5(tmp)

        # 解码器 (Decoder)
        
        # Block 1

        up_feature1 = mint.nn.functional.interpolate(
            feature5, 
            size=feature4.shape[2:],  # 自动获取 feature4 的高和宽
            mode='bilinear', 
            align_corners=True
        )
        tmp = mint.cat((feature4, up_feature1), dim=1) 
        tmp = self.double_conv6(tmp)
        
        # Block 2
        up_feature2 = mint.nn.functional.interpolate(
            tmp, 
            size=feature3.shape[2:], 
            mode='bilinear', 
            align_corners=True
        )
        tmp = mint.cat((feature3, up_feature2), dim=1)
        tmp = self.double_conv7(tmp)
        
        # Block 3
        up_feature3 = mint.nn.functional.interpolate(
            tmp, 
            size=feature2.shape[2:], 
            mode='bilinear', 
            align_corners=True
        )
        tmp = mint.cat((feature2, up_feature3), dim=1)
        tmp = self.double_conv8(tmp)
        
        # Block 4
        up_feature4 = mint.nn.functional.interpolate(
            tmp, 
            size=feature1.shape[2:], 
            mode='bilinear', 
            align_corners=True
        )
        tmp = mint.cat((feature1, up_feature4), dim=1)
        tmp = self.double_conv9(tmp)
        
        # Output
        # 使用  mint.sigmoid
        # output = mint.sigmoid(self.final(tmp))
        
        # return output
        return self.final(tmp)

# 实例化模型
print("Re-initializing UNet...")
net = UNet()

#### 2.4 自定义评估指标

为了能够更加全面和直观的观察网络模型训练效果，本案例实现中还使用了MindSpore框架来自定义Metrics，在自定义的metrics类中使用了多种评价函数来评估模型的好坏，分别为准确率Acc、交并比IoU、Dice系数、灵敏度Sens、特异性Spec。

a) 其中准确率Acc是图像中正确分类的像素百分比。即分类正确的像素占总像素的比例，用公式可表示为：
$$
A c c=\frac{T P+T N}{T P+T N+F P+F N}
$$
其中：

- TP：真阳性数，在label中为阳性，在预测值中也为阳性的个数。
- TN：真阴性数，在label中为阴性，在预测值中也为阴性的个数。
- FP：假阳性数，在label中为阴性，在预测值中为阳性的个数。
- FN：假阴性数，在label中为阳性，在预测值中为阴性的个数。

b) 交并比IoU是预测分割和标签之间的重叠区域除以预测分割和标签之间的联合区域（两者的交集/两者的并集），是语义分割中最常用的指标之一，其计算公式为：
$$
I o U=\frac{|A \cap B|}{|A \cup B|}=\frac{T P}{T P+F P+F N}
$$
c) Dice系数定义为两倍的交集除以像素和，也叫F1 score，与IoU呈正相关关系，其计算公式为：
$$
\text { Dice }=\frac{2|A \cap B|}{|A|+|B|}=\frac{2 T P}{2 T P+F P+F N}
$$
d) 敏感度Sens和特异性Spec分别是描述识别出的阳性占所有阳性的比例，以及描述识别出的负例占所有负例的比例，计算公式分别为：
$$
\text { Sens }=\frac{T P}{T P+F N}
$$

$$
\text { Spec }=\frac{T N}{F P+T N}
$$

具体的实现方法首先是自定义metrics_类，并按照MindSpore官方文档继承nn.Metric父类，接着根据上述5个评价指标的计算公式，在类中定义5个指标的计算方法，之后通过重新实现clear方法来初始化相关参数；重新实现update方法来传入模型预测值和标签，通过上述定义的各评价指标计算方法，计算每个指标的值并存入一个列表；最后通过重新实现eval方法来讲存储各评估指标值的列表返回。上述流程对应的代码如下：


In [ ]:
import mindspore as ms
from mindspore import nn, mint, Tensor

class metrics_(nn.Metric):
    def __init__(self, metrics, smooth=1e-5):
        """
        初始化
        metrics: list, e.g. ["acc", "iou", "dice", "sens", "spec"]
        """
        super(metrics_, self).__init__()
        self.metrics = metrics
        self.smooth = float(smooth) # 简单类型转换，替代 Validator
        self.metrics_list = [0. for i in range(len(self.metrics))]
        self._samples_num = 0
        self.clear()

    def clear(self):
        """清除内部评估结果"""
        self.metrics_list = [0. for i in range(len(self.metrics))]
        self._samples_num = 0

    # 以下所有 Metrics 方法均接收 Tensor 输入，使用 mint 算子计算
    # 使用 .item() 将单步结果转为 Python float 累加，保持原逻辑的数值精度
    
    def Acc_metrics(self, y_pred, y):
        # 展平
        y_pred_f = y_pred.view(-1)
        y_f = y.view(-1)
        
        # 计算相等元素的个数
        tp = mint.sum(mint.eq(y_pred_f, y_f)).float()
        total = float(len(y_pred_f))
        
        single_acc = tp / total
        return single_acc.item()

    def IoU_metrics(self, y_pred, y):
        y_pred_f = y_pred.view(-1)
        y_f = y.view(-1)
        
        intersection = mint.sum(y_pred_f * y_f)
        # Union = A + B - Intersection
        unionset = mint.sum(y_pred_f) + mint.sum(y_f) - intersection
        
        single_iou = intersection / (unionset + self.smooth)
        return single_iou.item()

    def Dice_metrics(self, y_pred, y):
        y_pred_f = y_pred.view(-1)
        y_f = y.view(-1)
        
        intersection = mint.sum(y_pred_f * y_f)
        unionset = mint.sum(y_pred_f) + mint.sum(y_f)
        
        single_dice = 2 * intersection / (unionset + self.smooth)
        return single_dice.item()

    def Sens_metrics(self, y_pred, y):
        y_pred_f = y_pred.view(-1)
        y_f = y.view(-1)
        
        tp = mint.sum(y_pred_f * y_f)
        actual_positives = mint.sum(y_f)
        
        single_sens = tp / (actual_positives + self.smooth)
        return single_sens.item()

    def Spec_metrics(self, y_pred, y):
        y_pred_f = y_pred.view(-1)
        y_f = y.view(-1)
        
        # TN: (1-pred) * (1-y)
        true_neg = mint.sum((1 - y_f) * (1 - y_pred_f))
        total_neg = mint.sum(1 - y_f)
        
        single_spec = true_neg / (total_neg + self.smooth)
        return single_spec.item()

    def update(self, *inputs):
        if len(inputs) != 2:
            raise ValueError("For 'update', it needs 2 inputs (predicted value, true value), "
                             "but got {}.".format(len(inputs)))
        
        y_pred_raw = inputs[0]
        y_true = inputs[1]

        # 确保数据类型一致，通常转为 float32 进行计算
        y_true = y_true.float()
        
        # 使用 mint 算子进行二值化
        y_pred = (y_pred_raw > 0.5).float()

        if y_pred.shape != y_true.shape:
            raise ValueError(f"For 'update', shapes must match. "
                             f"Pred: {y_pred.shape}, True: {y_true.shape}.")

        batch_size = y_true.shape[0]
        self._samples_num += batch_size

        # 逐样本计算指标
        for i in range(batch_size):
            if "acc" in self.metrics:
                self.metrics_list[0] += self.Acc_metrics(y_pred[i], y_true[i])
            if "iou" in self.metrics:
                self.metrics_list[1] += self.IoU_metrics(y_pred[i], y_true[i])
            if "dice" in self.metrics:
                self.metrics_list[2] += self.Dice_metrics(y_pred[i], y_true[i])
            if "sens" in self.metrics:
                self.metrics_list[3] += self.Sens_metrics(y_pred[i], y_true[i])
            if "spec" in self.metrics:
                self.metrics_list[4] += self.Spec_metrics(y_pred[i], y_true[i])

    def eval(self):
        if self._samples_num == 0:
            raise RuntimeError("Samples number is 0, please call update before eval.")
            
        # 计算平均值
        return [val / float(self._samples_num) for val in self.metrics_list]

In [ ]:
x = Tensor(np.array([[[[0.2, 0.5, 0.7], [0.3, 0.1, 0.2], [0.9, 0.6, 0.8]]]]))
y = Tensor(np.array([[[[0, 1, 1], [1, 0, 0], [0, 1, 1]]]]))
metric = metrics_(["acc", "iou", "dice", "sens", "spec"],smooth=1e-5)
metric.clear()
metric.update(x, y)
res = metric.eval()
print( '丨acc: %.4f丨丨iou: %.4f丨丨dice: %.4f丨丨sens: %.4f丨丨spec: %.4f丨' % (res[0], res[1], res[2], res[3],res[4]), flush=True)

#### 2.5 模型训练及评估

在模型训练时，首先是设置模型训练的epoch次数为50，再通过2.1节中自定义的create_dataset方法创建了训练集和验证集，其中训练集batch_size大小为4，验证集batch_size大小为2，图像尺寸统一调整为224x224；损失函数使用nn.BCELoss，优化器使用nn.Adam，并设置学习率为0.01。回调函数方面使用了LossMonitor和TimeMonitor来监控训练过程中每个epoch结束后，损失值Loss的变化情况以及每个epoch、每个step的运行时间，还实例化了2.5节中自定义的回调类EvalCallBack，实现计算每个epoch结束后，在2.4节中定义的5个评估指标，并保存当前最优模型。在50个epcoh结束后，模型在训练集和验证集上的评估指标如表1所示：

模型训练部分的代码如下：

In [ ]:
import os
import mindspore as ms
from mindspore import nn, mint, value_and_grad
import ml_collections

# 1. 设置动态图模式
ms.set_context(mode=ms.PYNATIVE_MODE)

def get_config():
    """configuration """
    config = ml_collections.ConfigDict()
    config.epochs = 100
    # 请确保路径正确
    config.train_data_path = "datasets/ISBI/train/"
    config.val_data_path = "datasets/ISBI/val/"
    config.imgsize = 224
    config.batch_size = 4
    config.pretrained_path = None
    config.in_channel = 3
    config.n_classes = 1
    config.lr = 0.001
    return config

cfg = get_config()

train_dataset = create_dataset(cfg.train_data_path, img_size=cfg.imgsize, batch_size= cfg.batch_size, augment=True, shuffle = True)
val_dataset = create_dataset(cfg.val_data_path, img_size=cfg.imgsize, batch_size= cfg.batch_size, augment=False, shuffle = False)

def train(model, dataset, loss_fn, optimizer, met):
    # 定义正向计算函数
    def forward_fn(data, label):
        logits = model(data)
        loss = loss_fn(logits, label)
        return loss, logits

    # 获取梯度计算函数 
    grad_fn = value_and_grad(forward_fn, None, optimizer.parameters, has_aux=True)

    size = dataset.get_dataset_size()
    model.set_train(True)
    
    # 实例化指标计算器 
    metric = metrics_(met, smooth=1e-5)
    metric.clear()
    
    train_loss = 0

    # 使用迭代器
    iterator = dataset.create_tuple_iterator()
    
    for batch, (data, label) in enumerate(iterator):
        # 1. 计算梯度
        (loss, logits), grads = grad_fn(data, label)
        current_loss = loss.item()
        
        # 2. 更新指标
        metric.update(logits, label)

        # 3. 优化器更新参数 
        optimizer(grads)
        
        # 4. 累计 Loss (用于显示)
        train_loss += current_loss
        

    train_loss /= size
    
    # 5. 计算最终指标
    res = metric.eval()
    print(f'Train loss:{train_loss:>4f}','丨acc: %.3f丨丨iou: %.3f丨丨dice: %.3f丨丨sens: %.3f丨丨spec: %.3f丨' % (res[0], res[1], res[2], res[3], res[4]))


def val(model, dataset, loss_fn, met):
    size = dataset.get_dataset_size()
    model.set_train(False)
    
    # 实例化指标计算器
    metric = metrics_(met, smooth=1e-5)
    metric.clear()
    
    val_loss = 0

    iterator = dataset.create_tuple_iterator()
    
    for batch, (data, label) in enumerate(iterator):
        # 1. 前向推理
        logits = model(data)
        
        # 2. 计算 Loss
        loss = loss_fn(logits, label)
        val_loss += loss.item()
        
        # 3. 更新指标 (Batch-wise update)
        metric.update(logits, label)

    val_loss /= size
    
    # 4. 计算最终指标
    res = metric.eval()

    print(f'Val loss:{val_loss:>4f}','丨acc: %.3f丨丨iou: %.3f丨丨dice: %.3f丨丨sens: %.3f丨丨spec: %.3f丨' % (res[0], res[1], res[2], res[3], res[4]))

    checkpoint = res[1] 
    return checkpoint, res[4]

# --- 主程序部分 ---

# 1. 实例化模型
net = UNet(in_ch=cfg.in_channel, n_classes=cfg.n_classes)

# 2. 定义 Loss 
criterion = mint.nn.BCEWithLogitsLoss()

# 3. 定义优化器 
optimizer = mint.optim.SGD(params=net.trainable_params(), lr=cfg.lr)

# 获取步数信息
iters_per_epoch = train_dataset.get_dataset_size()
total_train_steps = iters_per_epoch * cfg.epochs
print('iters_per_epoch: ', iters_per_epoch)
print('total_train_steps: ', total_train_steps)

metrics_name = ["acc", "iou", "dice", "sens", "spec"]

best_iou = 0
ckpt_path = 'checkpoint/best_UNet.ckpt'

# 创建保存目录
if not os.path.exists("checkpoint"):
    os.makedirs("checkpoint")

for epoch in range(cfg.epochs):
    print(f"Epoch [{epoch+1} / {cfg.epochs}]")
    
    # 训练
    train(net, train_dataset, criterion, optimizer, metrics_name)
    
    # 验证
    checkpoint_best, spec = val(net, val_dataset, criterion, metrics_name)
    
    # 保存
    if epoch > 2 and spec > 0.2:
        if checkpoint_best > best_iou:
            print('IoU improved from %0.4f to %0.4f' % (best_iou, checkpoint_best))
            best_iou = checkpoint_best
            ms.save_checkpoint(net, ckpt_path)
            print("saving best checkpoint at: {} ".format(ckpt_path))
        else:
            print('IoU did not improve from %0.4f' % (best_iou),"\n-------------------------------")
            
print("Done!")

#### 2.6 模型预测
在预测部分需要创建一个测试数据集，放在datasets/ISBI/test/文件夹下，代码如下：

In [ ]:
import os
import cv2
import glob
import random
import numpy as np
from tqdm import tqdm

import mindspore
import mindspore.dataset as ds
import mindspore.dataset.vision as vision
import mindspore.dataset.transforms as transforms
from mindspore import mint, ops, Tensor



def val_transforms(img_size):
    """
    使用标准的 mindspore.dataset.vision 接口
    """
    return transforms.Compose([
        vision.Resize(img_size, interpolation=vision.Inter.NEAREST),
        vision.Rescale(1/255., 0),
        vision.HWC2CHW()
    ])

class Data_Loader:
    def __init__(self, data_path, have_mask):
        self.data_path = data_path
        self.have_mask = have_mask
        self.imgs_path = glob.glob(os.path.join(data_path, 'image/*.png'))
        if self.have_mask:
            self.label_path = glob.glob(os.path.join(data_path, 'mask/*.png'))

    def __getitem__(self, index):
        image = cv2.imread(self.imgs_path[index])
        if self.have_mask:
            label = cv2.imread(self.label_path[index], cv2.IMREAD_GRAYSCALE)
            label = label.reshape((label.shape[0], label.shape[1], 1))
        else:

            label = image 
        return image, label

    @property
    def column_names(self):
        return ['image', 'label']

    def __len__(self):
        return len(self.imgs_path)

def create_dataset(data_dir, img_size, batch_size, shuffle, have_mask=False):
    mc_dataset = Data_Loader(data_path=data_dir, have_mask=have_mask)
    print(f"Dataset size: {len(mc_dataset)}")
    
    # 定义数据集
    dataset = ds.GeneratorDataset(mc_dataset, mc_dataset.column_names, shuffle=shuffle)
    
    # 预处理操作
    transform_img = val_transforms(img_size)
    
    # 设置随机种子
    seed = random.randint(1, 1000)
    ds.config.set_seed(seed) # 更新为推荐的设置种子方式
    
    # Map 操作
    dataset = dataset.map(input_columns='image', num_parallel_workers=1, operations=transform_img)
    dataset = dataset.map(input_columns="label", num_parallel_workers=1, operations=transform_img)
    
    dataset = dataset.batch(batch_size, num_parallel_workers=1)
    return dataset

def model_pred(model, test_loader, result_path, have_mask):
    """
    使用 MindSpore 2.7.1+ 的 mint 接口进行预测后处理
    """
    model.set_train(False)
    test_pred = []
    test_label = []
    
    if not os.path.exists(result_path):
        os.makedirs(result_path)

    print("Start Prediction...")
    # 使用 create_tuple_iterator 获取数据
    for batch_idx, (data, label) in enumerate(tqdm(test_loader.create_tuple_iterator())):
        
        # 1. 模型推理 (输出通常为 Logits 或 Probabilities)
        logits = model(data)
        
        # 2. 使用 mint 接口进行二值化处理

        pred_binary = (logits > 0.5).float() 

        # 3. 维度变换 (Tensor操作)

        img_tensor = mint.squeeze(pred_binary, dim=0)
        
        # Mint: permute 调整通道顺序 (C, H, W) -> (H, W, C)
        img_tensor = img_tensor.permute(1, 2, 0)

        # 4. 转为 Numpy 进行保存 (cv2 需要 numpy)

        img_np = img_tensor.asnumpy() * 255.0
        
        # 保存结果
        cv2.imwrite(os.path.join(result_path, "%05d.png" % batch_idx), img_np)

        # 5. 收集结果用于指标计算 (转为扁平列表)
 
        test_pred.extend(pred_binary.asnumpy().flatten())
        test_label.extend(label.asnumpy().flatten())

    if have_mask:

        mtr = ['acc', 'iou', 'dice', 'sens', 'spec']
        try:

            metric = metrics_(mtr, smooth=1e-5) 
            metric.clear()
            metric.update(test_pred, test_label)
            res = metric.eval()
            print(f'丨acc: %.3f丨丨iou: %.3f丨丨dice: %.3f丨丨sens: %.3f丨丨spec: %.3f丨' % 
                  (res[0], res[1], res[2], res[3], res[4]))
        except NameError:
            print("Warning: 'metrics_' class is not defined. Skipping metric evaluation.")
    else:
        print("Evaluation metrics cannot be calculated without Mask")


net = UNet(3, 1)
# 加载权重
param_dict = mindspore.load_checkpoint("checkpoint/best_UNet.ckpt")
mindspore.load_param_into_net(net, param_dict)
        
result_path = "predict"
# 创建测试集
test_dataset = create_dataset("datasets/ISBI/test/", 224, 1, shuffle=False, have_mask=False)
        
# 执行预测
model_pred(net, test_dataset, result_path, have_mask=False)

#### 2.7 可视化预测结果

In [ ]:
image_path = "datasets/ISBI/test/image/"
pred_path = "predict/"

image_list = os.listdir(image_path)
pred_list = os.listdir(pred_path)[1:]
# print(image_list)
# print(pred_list)
test_image = np.array([cv2.imread(image_path + image_list[p], -1) for p in range(len(image_list))])
pred_masks = np.array([cv2.imread(pred_path + pred_list[p], -1) for p in range(len(pred_list))])

show_image(test_image, num = 12)
show_image(pred_masks, num = 12)

```

|                | Acc  | IoU  | Dice | Sens | Spec |
| :------------: | :--: | :--: | :--: | ---: | ---- |
|   Train set    | 0.90 | 0.88 | 0.94 | 0.94 | 0.76 |
| Validation set | 0.92 | 0.90 | 0.95 | 0.96 | 0.77 |
```

根据表1评价指标结构，本案例构建的网络模型具有较好的性能，能够实现对测试集进行较为准确的预测，针对测试集的部分预测结果如图4所示。

<center>
    <img src="https://tva1.sinaimg.cn/large/e6c9d24ely1h5bwigmy47j20l2064dgy.jpg" alt="image-20220819101847606" style="zoom:50%;" />
    <br>
    <div style="color:orange;
    display: inline-block;
    color: #999;
    padding: 2px;">图4 模型预测结果</div>
</center>

### 3 总结

本案例基于MindSpore框架针对ISBI数据集，完成了数据读取、数据集创建、Unet模型构建，并根据特定需求自定义了评估指标和回调函数，进行了模型训练和评估，顺利完成了预测结果的输出。通过此案例进一步加深了对Unet模型结构和特性的理解，并结合MindSpore框架提供的文档和教程，掌握了利用Mindspore框架实现特定案例的流程，以及多种API的使用方法，为以后在实际场景中应用MindSpore框架提供支持。